In [2]:
import pandas as pd
import os

# Set the working directory
data_dir = '/Users/antho/Documents/WPI-MW'
os.chdir(data_dir)

# Filenames
donor_file = 'DonationsLatest.xlsx'
patron_file = 'summary_Patrons.xlsx'

# Step 1: Read the two files
df = pd.read_excel(donor_file, parse_dates=['Close Date'])
donor_df = df.copy()

# Ensure both columns are datetime
donor_df['Last Donation Date'] = pd.to_datetime(donor_df['Last Donation Date'], errors='coerce')
donor_df['Close Date'] = pd.to_datetime(donor_df['Close Date'], errors='coerce')

# Fill missing 'Last Donation Date' with 'Close Date'
donor_df['Last Donation Date'] = donor_df['Last Donation Date'].fillna(donor_df['Close Date'])

print('donor file read')


patron_df = pd.read_excel(patron_file)
print('patron file read')

# Step 2: Merge with an indicator to show merge source
patron_details_df = pd.merge(
    patron_df,
    donor_df.drop(columns='Account ID', errors='ignore'),
    on='Account Name',
    how='left',
    indicator='merge_status',
    validate='1:m'
)
print('patron and donor details file merged')

# Step 1: Clean donation columns
donations_df = patron_details_df[['Account Name', 'Amount', 'Close Date']].copy()
donations_df = donations_df.dropna(subset=['Amount', 'Close Date'])
donations_df['Close Date'] = pd.to_datetime(donations_df['Close Date'], errors='coerce')


# Step 2: Aggregate donation stats per Account
donor_summary_df = (
    donations_df
    .groupby('Account Name', as_index=False)
    .agg(
        total_amount=('Amount', 'sum'),
        avg_amount=('Amount', 'mean'),
        donation_count=('Amount', 'count'),
        first_date=('Close Date', 'min'),
        last_date=('Close Date', 'max')
    )
    .rename(columns={
        'total_amount': 'Lifetime Donations',
        'avg_amount': 'Average Donation',
        'donation_count': 'Donation Count',
        'first_date': 'First Donation Date',
        'last_date': 'Last Donation Date'}
    )
)
print('donor aggregations complete')



# After merging donor_summary_df into patron_details_df:
patron_summary_df = pd.merge(
    patron_details_df.drop(columns='Last Donation Date'), # redundant column
    donor_summary_df,
    on='Account Name',
    how='left')

# Ensure numeric fields are filled with 0 if missing
patron_summary_df[['Lifetime Donations', 'Average Donation', 'Donation Count']] = (
    patron_summary_df[['Lifetime Donations', 'Average Donation', 'Donation Count']].fillna(0)
)

# Ensure dates are datetime and show in short format
patron_summary_df['First Donation Date'] = pd.to_datetime(patron_summary_df['First Donation Date'], errors='coerce').dt.date
patron_summary_df['Last Donation Date'] = pd.to_datetime(patron_summary_df['Last Donation Date'], errors='coerce').dt.date



# Step 3: Merge with basic patron info
patron_summary_df = patron_summary_df.drop_duplicates(subset='Account Name')[
    ['Account Name',
     'Customer Segment',
     'RFM Score',
     'Recency (Months)',
     'Frequency',
     'Customer Lifespan',
     'Average Yearly Monetary',
     'Growth Score',
     'Regularity',
     'First Donation Date',
     'Last Donation Date',
     'Lifetime Donations',
     'Average Donation',
     'Donation Count',
     'Is Subscriber',
     'Patron Status',
     'In Chorus',
     'Favorite Genre',
     'Genre Strength',
     'Favorite Class',
     'First Event',
     'Most Recent Event',
     'Geo Region',
     'Account ID',
     'Contact ID']
]
print('output prep complete complete')

# Step 4: Write both DataFrames to Excel
output_file = 'Patron Summary.xlsx'
#with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    patron_summary_df.to_excel(writer, index=False, sheet_name='Patron Summary')
    patron_details_df.to_excel(writer, index=False, sheet_name='Patron Details')

print(f"Excel file with summary + full data written to: {output_file}")

donor file read
patron file read
patron and donor details file merged
donor aggregations complete
output prep complete complete
Excel file with summary + full data written to: Patron Summary.xlsx
